# SPY 0DTE Candle-Pattern Strategy

Scans 10 candle patterns on 5-min bars. **First pattern that fires with all 4 indicators confirming wins.** One trade/day, no Fridays.

**Patterns (5 bull / 5 bear):**
1. Three White Soldiers / Three Black Crows
2. Morning Star / Evening Star
3. Bullish / Bearish Engulfing + confirmation
4. Hammer / Shooting Star + confirmation
5. Piercing / Dark Cloud Cover + confirmation

**Confirmations required on the detection bar's close (all 4):**
- Calls: close > VWAP, EMA9 > EMA21, close > EMA9, RSI > 50
- Puts: close < VWAP, EMA9 < EMA21, close < EMA9, RSI < 50

**Exits:** 10% gross profit, 20% stop, time-stop (sweepable). Trade ATM 0DTE call/put.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Sweep over the cached year (9 configs ≈ 2-3 minutes)

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 4. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. Per-pattern performance — which patterns actually have edge?

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
if not taken.empty:
    print('=== By pattern ===')
    print(taken.groupby('pattern').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        median_pnl=('net_pnl', 'median'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2).sort_values('total_pnl', ascending=False))
    print('\n=== By direction ===')
    print(taken.groupby('signal_direction').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))

## 6. Timing

In [ ]:
from iron_condor.metrics import analyze_timing
print(analyze_timing(trades))

## 7. Equity curve — top config

In [ ]:
import matplotlib.pyplot as plt
top = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(10, 5))
for cfg in top:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Top-3 candle-pattern config equity curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()